In [1]:
import requests
import pandas as pd
import json
from dateutil import parser

In [2]:
API_KEY = "25ca3f1c80e0bd573abf125260d832d8-b9884a5701c2ca83d3a7a00e0fd3a1f0"
ACCOUNT_ID = "101-011-30426832-001"
OANDA_URL = "https://api-fxpractice.oanda.com/v3"

In [3]:
session = requests.Session()

In [4]:
session.headers.update({
    "Authorization": f"Bearer {API_KEY}",
    "Content_Type": "application/json"
})

In [5]:
params = dict(
    count = 10,
    granularity = "H1",
    price = "MBA"
)

In [6]:
url = f"{OANDA_URL}/accounts/{ACCOUNT_ID}/instruments"

In [7]:
res = session.get(url, params=None, data=None, headers=None)

In [8]:
print("Status Code:", res.status_code)
data = res.json()

Status Code: 200


In [9]:
instruments_list = data['instruments']
print("Instruments length:", len(instruments_list))

Instruments length: 127


In [10]:
instruments_list[0].keys()

dict_keys(['name', 'type', 'displayName', 'pipLocation', 'displayPrecision', 'tradeUnitsPrecision', 'minimumTradeSize', 'maximumTrailingStopDistance', 'minimumTrailingStopDistance', 'maximumPositionSize', 'maximumOrderUnits', 'marginRate', 'guaranteedStopLossOrderMode', 'tags', 'financing'])

In [11]:
key_i = ['name', 'type', 'displayName', 'pipLocation', 'displayPrecision', 'tradeUnitsPrecision', 'marginRate']

In [20]:
instruments_dict = {}
for instr in instruments_list:
    key = instr['name']
    instruments_dict[key] = { k: instr[k] for k in key_i }

instruments_dict['USD_CAD']

{'name': 'USD_CAD',
 'type': 'CURRENCY',
 'displayName': 'USD/CAD',
 'pipLocation': -4,
 'displayPrecision': 5,
 'tradeUnitsPrecision': 0,
 'marginRate': '0.0333'}

In [19]:
# pipLocation: -4 -> 0.0001
pow(10, -4)

0.0001

In [13]:
with open("../data/instruments.json", "w") as f:
    f.write(json.dumps(instruments_dict, indent=3))

In [14]:
def fetch_candles(pair_name, count=10, granularity="H1"):
    url = f"{OANDA_URL}/instruments/{pair_name}/candles"
    params = dict(
        count = count,
        granularity = granularity,
        price = "MBA"
    )
    res = session.get(url, params=params, data=None, headers=None)
    data = res.json()
    

    if res.status_code == 200:
        if 'candles' not in data:
            data = []
        else:
            data = data['candles']

    return res.status_code, data

def get_candles_df(data):
    if len(data) == 0:
        return pd.DataFrame()
    
    prices = ['mid', 'bid', 'ask']
    ohlc = ['o', 'h', 'l', 'c']

    final_data = []
    for candle in data:
        if candle['complete'] == False:
            continue
        new_dict = {}
        new_dict['time'] = parser.parse(candle['time'])

        new_dict['volume'] = candle['volume']
        for p in prices:
            for o in ohlc:
                new_dict[f"{p}_{o}"] = float(candle[p][o])
        final_data.append(new_dict)

    # Since the last candle in incomplete, it would not be included
    # Thus, total candles = count-1
    df = pd.DataFrame.from_dict(final_data)
    return df

def create_data_file(pair_name, count=10, granularity="H1"):
    code, data = fetch_candles(pair_name, count, granularity)
    if code != 200:
        print("Failed", pair_name, data)
        return
    if len(data) == 0:
        print("No candles", pair_name)
    candles_df = get_candles_df(data)
    candles_df.to_pickle(f"../data/{pair_name}_{granularity}.pkl")
    print(f"{pair_name} {granularity} {candles_df.shape[0]} candles, {candles_df.time.min()} {candles_df.time.max()}")

In [15]:
code, data = fetch_candles("EUR_USD", count=10, granularity="H4")
candles_df = get_candles_df(data)
candles_df

,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c
0,2024-11-25 18:00:00+00:00,17994,1.04861,1.05108,1.04856,1.04966,1.04854,1.05101,1.04849,1.04956,1.04868,1.05115,1.04863,1.04975
1,2024-11-25 22:00:00+00:00,36351,1.04942,1.05008,1.04250,1.04541,1.04904,1.04999,1.04242,1.04533,1.04979,1.05018,1.04258,1.04549
2,2024-11-26 02:00:00+00:00,18926,1.04542,1.04906,1.04512,1.04753,1.04533,1.04898,1.04504,1.04745,1.04550,1.04914,1.04521,1.04761
3,2024-11-26 06:00:00+00:00,39886,1.04752,1.05193,1.04700,1.05073,1.04744,1.05185,1.04691,1.05065,1.04759,1.05201,1.04708,1.05081
4,2024-11-26 10:00:00+00:00,43858,1.05072,1.05448,1.05000,1.05036,1.05065,1.05441,1.04993,1.05029,1.05080,1.05457,1.05007,1.05044
5,2024-11-26 14:00:00+00:00,48928,1.05037,1.05102,1.04636,1.04705,1.05030,1.05094,1.04629,1.04698,1.05044,1.05109,1.04643,1.04712
6,2024-11-26 18:00:00+00:00,23098,1.04704,1.04904,1.04573,1.04894,1.04697,1.04897,1.04566,1.04885,1.04712,1.04911,1.04580,1.04903
7,2024-11-26 22:00:00+00:00,15885,1.04876,1.04965,1.04844,1.04868,1.04848,1.04958,1.04836,1.04860,1.04903,1.04973,1.04852,1.04875
8,2024-11-27 02:00:00+00:00,12868,1.04867,1.04883,1.04744,1.04770,1.04859,1.04875,1.04737,1.04762,1.04875,1.04891,1.04752,1.04777


In [16]:
create_data_file("EUR_USD", count=10, granularity="H4")

EUR_USD H4 9 candles, 2024-11-25 18:00:00+00:00 2024-11-27 02:00:00+00:00


In [17]:
our_curr = ['EUR', 'USD', 'GBP', 'JPY', 'CHF', 'NZD', 'CAD', 'AUD']
instruments_dict.keys()

dict_keys(['XAG_SGD', 'AUD_NZD', 'BCO_USD', 'NZD_USD', 'CORN_USD', 'NL25_EUR', 'CAD_JPY', 'USD_ZAR', 'SG30_SGD', 'EUR_USD', 'SOYBN_USD', 'XAU_EUR', 'XPT_USD', 'USD_DKK', 'AU200_AUD', 'XAU_XAG', 'XAU_GBP', 'NAS100_USD', 'GBP_AUD', 'USD_PLN', 'CHINAH_HKD', 'CH20_CHF', 'CAD_HKD', 'BCH_USD', 'XAG_CHF', 'USD_CHF', 'XAG_HKD', 'AUD_HKD', 'ESPIX_EUR', 'NZD_CHF', 'AUD_CHF', 'GBP_CHF', 'USD_THB', 'XAU_JPY', 'XAU_HKD', 'EUR_HKD', 'CHF_JPY', 'GBP_HKD', 'EUR_NZD', 'XAG_AUD', 'WTICO_USD', 'XAG_NZD', 'AUD_SGD', 'EUR_JPY', 'EUR_TRY', 'USD_JPY', 'BTC_USD', 'SGD_JPY', 'GBP_ZAR', 'XAG_JPY', 'ETH_USD', 'ZAR_JPY', 'NZD_SGD', 'EUR_DKK', 'USD_HUF', 'HKD_JPY', 'DE30_EUR', 'US2000_USD', 'NATGAS_USD', 'DE10YB_EUR', 'GBP_CAD', 'UK100_GBP', 'EUR_HUF', 'USD_SEK', 'GBP_SGD', 'XPD_USD', 'XAU_CHF', 'XAU_CAD', 'EUR_PLN', 'SUGAR_USD', 'AUD_CAD', 'USB05Y_USD', 'UK10YB_GBP', 'EUR_CAD', 'USD_MXN', 'GBP_USD', 'CAD_SGD', 'XAG_CAD', 'JP225_USD', 'FR40_EUR', 'USB30Y_USD', 'NZD_HKD', 'XAG_USD', 'EUR_CZK', 'EUR_CHF', 'WHEAT_USD

In [18]:
for c1 in our_curr:
    for c2 in our_curr:
        pair = f"{c1}_{c2}"
        if pair in instruments_dict:
            for g in ["H1", "H4"]:
                create_data_file(pair, count=4001, granularity=g)

EUR_USD H1 4000 candles, 2024-04-08 13:00:00+00:00 2024-11-27 07:00:00+00:00
EUR_USD H4 4000 candles, 2022-05-04 13:00:00+00:00 2024-11-27 02:00:00+00:00
EUR_GBP H1 4000 candles, 2024-04-08 13:00:00+00:00 2024-11-27 07:00:00+00:00
EUR_GBP H4 4000 candles, 2022-05-04 13:00:00+00:00 2024-11-27 02:00:00+00:00
EUR_JPY H1 4000 candles, 2024-04-08 13:00:00+00:00 2024-11-27 07:00:00+00:00
EUR_JPY H4 4000 candles, 2022-05-04 05:00:00+00:00 2024-11-27 02:00:00+00:00
EUR_CHF H1 4000 candles, 2024-04-08 13:00:00+00:00 2024-11-27 07:00:00+00:00
EUR_CHF H4 4000 candles, 2022-05-04 13:00:00+00:00 2024-11-27 02:00:00+00:00
EUR_NZD H1 4000 candles, 2024-04-08 13:00:00+00:00 2024-11-27 07:00:00+00:00
EUR_NZD H4 4000 candles, 2022-05-04 21:00:00+00:00 2024-11-27 02:00:00+00:00
EUR_CAD H1 4000 candles, 2024-04-08 13:00:00+00:00 2024-11-27 07:00:00+00:00
EUR_CAD H4 4000 candles, 2022-05-04 13:00:00+00:00 2024-11-27 02:00:00+00:00
EUR_AUD H1 4000 candles, 2024-04-08 13:00:00+00:00 2024-11-27 07:00:00+00:00